In [1]:
# [MARKDOWN]
# ## FN audit — 3 steps
# 1. **Count misses** — box in consensus + model prob < threshold
# 2. **Split misses** — IG for the *missed class*: 👁️ attentive (peak in box OR mass≥10%) vs 🙈 blind
# 3. **Clinical danger** — aggregate by Critical / Moderate / Mild tier
#
# **GPU save:** reuse NB04 `top50_path` when `ig_manifest.target_class == missed_class`; new IG only otherwise (~85% of FNs).



In [2]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())



Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


In [3]:
!pip install -q torch==2.1.0 torchvision==0.16.0 timm==0.9.12 captum==0.7.0 peft==0.6.2 \
  scikit-learn==1.3.2 opencv-python-headless statsmodels==0.14.0 numpy==1.26.4

import torch
assert torch.cuda.is_available(), "GPU required"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")



ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.1.0
✅ GPU: Tesla T4


In [4]:
import os, gc, json, random, warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tvmodels
import timm
from torch.amp import autocast
from captum.attr import IntegratedGradients
from peft import LoraConfig, TaskType, get_peft_model

ROOT         = Path(GDRIVE_ROOT)
IMAGES_PATH  = ROOT / 'data' / 'processed' / 'images'
SPLITS_PATH  = ROOT / 'data' / 'processed' / 'splits'
ANN_PATH     = ROOT / 'data' / 'processed' / 'consensus'
MODELS_PATH  = ROOT / 'models'
RESULTS_PATH = ROOT / 'results'
FIGPATH      = ROOT / 'figures'
IG_FN_PATH   = ROOT / 'ig_maps' / 'fn'
IG_FN_TOP50  = IG_FN_PATH / 'top50'

for p in [RESULTS_PATH, FIGPATH, IG_FN_PATH, IG_FN_TOP50]:
    p.mkdir(parents=True, exist_ok=True)

IMG_SIZE      = 224
NUM_CLASSES   = 14
RANDOM_SEED   = 42
IG_STEPS      = 300
DELTA_THRESH  = 0.01
MASS_TAU      = 0.10          # bucket threshold for mass_in_box
SAVE_EVERY    = 25            # partial CSV flush interval
MODEL_NAMES   = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
CONSENSUS     = '2of3'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

SEVERITY_TIERS = {
    'Aortic enlargement': 'Critical', 'Pneumothorax': 'Critical',
    'Pleural effusion': 'Moderate', 'Consolidation': 'Moderate',
    'Atelectasis': 'Moderate', 'Cardiomegaly': 'Moderate',
    'Calcification': 'Mild', 'ILD': 'Mild', 'Infiltration': 'Mild',
    'Lung Opacity': 'Mild', 'Nodule/Mass': 'Mild', 'Other lesion': 'Mild',
    'Pleural thickening': 'Mild', 'Pulmonary fibrosis': 'Mild',
}
TIER_ORDER = ['Critical', 'Moderate', 'Mild']

print("✓ NB09 setup ready")



✓ NB09 setup ready


In [5]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def class_slug(name: str) -> str:
    return name.replace('/', '_').replace(' ', '_')

def find_box_columns(df):
    colmap = {}
    aliases = {
        'image_id': ['image_id', 'id'],
        'target_class': ['target_class', 'class', 'pathology', 'class_name'],
        'xmin': ['xmin', 'x1', 'left', 'x_min'],
        'ymin': ['ymin', 'y1', 'top', 'y_min'],
        'xmax': ['xmax', 'x2', 'right', 'x_max'],
        'ymax': ['ymax', 'y2', 'bottom', 'y_max'],
    }
    for target, opts in aliases.items():
        found = next((c for c in opts if c in df.columns), None)
        assert found, f'Missing {target} in {df.columns.tolist()}'
        colmap[target] = found
    return colmap

def box_dict(row, cols):
    return {
        'xmin': float(row[cols['xmin']]), 'ymin': float(row[cols['ymin']]),
        'xmax': float(row[cols['xmax']]), 'ymax': float(row[cols['ymax']]),
    }

def box_to_mask(box, h=IMG_SIZE, w=IMG_SIZE):
    m = np.zeros((h, w), dtype=bool)
    x1 = int(max(0, np.floor(box['xmin'])))
    y1 = int(max(0, np.floor(box['ymin'])))
    x2 = int(min(w, np.ceil(box['xmax'])))
    y2 = int(min(h, np.ceil(box['ymax'])))
    m[y1:y2, x1:x2] = True
    return m

def peak_in_box(attr_map, gt_box):
    """attr_map: (224,224) float, non-negative."""
    am = attr_map.copy()
    if am.ndim == 3:
        am = am.sum(axis=0)
    py, px = np.unravel_index(np.argmax(am), am.shape)
    return (gt_box['xmin'] <= px < gt_box['xmax']) and (gt_box['ymin'] <= py < gt_box['ymax'])

def mass_in_box(binary_mask, gt_box):
    m = binary_mask.astype(bool)
    gt = box_to_mask(gt_box)
    pos = m.sum()
    if pos == 0:
        return 0.0
    return float(np.logical_and(m, gt).sum() / pos)

def fn_bucket(peak, mass, tau=MASS_TAU):
    if peak or mass >= tau:
        return 'attentive'   # 👁️ saw it but chickened out
    return 'blind'           # 🙈 completely blind

def normalize_ig_map(attr):
    attr = np.abs(attr)
    if attr.ndim == 3:
        attr = attr.sum(axis=0)
    mn, mx = attr.min(), attr.max()
    if (mx - mn) < 1e-8:
        return np.zeros_like(attr, dtype=np.float32)
    return ((attr - mn) / (mx - mn)).astype(np.float32)

def top_mass_mask(attr_map, mass=0.5):
    flat = attr_map.flatten().astype(np.float32)
    order = np.argsort(flat)[::-1]
    csum = np.cumsum(flat[order])
    total = csum[-1]
    if total <= 0:
        return np.zeros_like(attr_map, dtype=np.uint8)
    cutoff = int(np.searchsorted(csum, mass * total, side='left')) + 1
    mask = np.zeros_like(flat, dtype=np.uint8)
    mask[order[:cutoff]] = 1
    return mask.reshape(attr_map.shape)

print('✓ helpers ready')



✓ helpers ready


In [6]:
def load_model(model_name: str, models_path: Path, selected_rank: int, num_classes: int) -> nn.Module:
    if model_name == 'densenet121':
        m = tvmodels.densenet121(weights=None)
        m.classifier = nn.Linear(1024, num_classes)
    elif model_name == 'convnextv2_tiny':
        m = timm.create_model(
            'convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=False,
            num_classes=0,
        )
        m.head.fc = nn.Linear(768, num_classes)
    elif model_name == 'swinb_lora':
        base = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=False,
            num_classes=num_classes,
        )
        lora_cfg = LoraConfig(
            r=selected_rank,
            lora_alpha=selected_rank * 2,
            target_modules=['qkv', 'proj'],
            lora_dropout=0.1,
            bias='none',
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        m = get_peft_model(base, lora_cfg)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    ckpt_path = models_path / f'{model_name}_finetuned.pt'
    assert ckpt_path.exists(), f"❌ Checkpoint not found: {ckpt_path}. Run NB02 first."
    state = torch.load(str(ckpt_path), map_location='cpu')

    state_keys = set(state.keys())
    lora_query_keys = {k for k in state_keys if 'query.lora' in k or 'value.lora' in k}
    if model_name == 'swinb_lora' and lora_query_keys:
        raise RuntimeError(
            f"❌ Checkpoint was trained with target_modules=['query','value'] (plan v7.0). "
            f"NB02 must be rerun with target_modules=['qkv','proj']. "
            f"Conflicting keys: {list(lora_query_keys)[:4]}"
        )

    m.load_state_dict(state)
    m = m.cuda().eval()
    if model_name == 'swinb_lora':
        m.forward = base.forward
    print(f"✅ {model_name} loaded from {ckpt_path.name}")
    return m

def run_ig(model: nn.Module, img_tensor: torch.Tensor, target_idx: int, n_steps: int):
    ig = IntegratedGradients(model)
    baseline = torch.zeros_like(img_tensor)
    attr, delta = ig.attribute(
            img_tensor,
            baselines=baseline,
            target=target_idx,
            n_steps=n_steps,
            internal_batch_size=50,
            return_convergence_delta=True,
        )
    return attr.detach().cpu().numpy()[0], float(delta.detach().cpu().item())

def compute_ig_with_convergence_check(model: nn.Module, img_tensor: torch.Tensor, target_idx: int):
    attr_raw, initial_delta = run_ig(model, img_tensor, target_idx, IG_STEPS)
    rerun = abs(initial_delta) > DELTA_THRESH
    steps_used = IG_STEPS
    final_delta = initial_delta

    if rerun:
        attr_raw, final_delta = run_ig(model, img_tensor, target_idx, 500)
        steps_used = 500

    attr_norm = normalize_ig_map(attr_raw)
    return attr_norm, initial_delta, final_delta, steps_used, rerun

test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

sweep_df = pd.read_csv(MODELS_PATH / 'lora_sweep.csv')
selected_rank = int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])

with open(MODELS_PATH / 'thresholds.json') as f:
    all_thresholds = json.load(f)

# LABEL_COLS: read header from test_patho (all cols except image_id)
_test = pd.read_csv(SPLITS_PATH / 'test_patho.csv', nrows=0)
LABEL_COLS = [c for c in _test.columns if c != 'image_id']
assert len(LABEL_COLS) == NUM_CLASSES



In [7]:
boxes = pd.read_csv(ANN_PATH / f'consensus_boxes_{CONSENSUS}.csv')
cols = find_box_columns(boxes)
boxes = boxes.rename(columns={cols['target_class']: 'missed_class'})
cols['target_class'] = 'missed_class'

test_patho = pd.read_csv(SPLITS_PATH / 'test_patho.csv')
test_patho['image_id'] = test_patho['image_id'].astype(str).str.replace('.png', '', regex=False)
test_ids = set(test_patho['image_id'])

boxes['image_id'] = boxes['image_id'].astype(str)
boxes = boxes[boxes['image_id'].isin(test_ids)].copy()

manifest = pd.read_csv(RESULTS_PATH / 'ig_manifest.csv')
manifest['image_id'] = manifest['image_id'].astype(str)
ig_lookup = manifest[manifest['subset'] == 'patho'].set_index(['image_id', 'model'])['target_class'].to_dict()
ig_top50_lookup = manifest[manifest['subset'] == 'patho'].set_index(['image_id', 'model'])['top50_path'].to_dict()

rows = []
for model in MODEL_NAMES:
    probs = pd.read_csv(RESULTS_PATH / f'{model}_test_image_probs.csv')
    probs['image_id'] = probs['image_id'].astype(str)
    thr = all_thresholds[model]

    merged = boxes.merge(probs, on='image_id', how='inner')
    for _, r in merged.iterrows():
        mc = r['missed_class']
        if mc not in thr or mc not in LABEL_COLS:
            continue
        p = float(r[mc])
        t = float(thr[mc])
        if p >= t:
            continue  # not FN

        ig_tgt = ig_lookup.get((r['image_id'], model))
        reusable = (ig_tgt == mc)
        rows.append({
            'image_id': r['image_id'],
            'model': model,
            'missed_class': mc,
            'tier': SEVERITY_TIERS.get(mc, 'Unknown'),
            'prob_miss': round(p, 6),
            'threshold': round(t, 6),
            'margin_below_thr': round(t - p, 6),
            'gt_xmin': r[cols['xmin']], 'gt_ymin': r[cols['ymin']],
            'gt_xmax': r[cols['xmax']], 'gt_ymax': r[cols['ymax']],
            'ig_target_nb04': ig_tgt,
            'ig_reusable': reusable,
            'nb04_top50_path': ig_top50_lookup.get((r['image_id'], model), ''),
        })

inv = pd.DataFrame(rows)
inv.to_csv(RESULTS_PATH / 'fn_inventory.csv', index=False)

print(f'✓ fn_inventory.csv: {len(inv)} FN events')
print('Reusable:', inv['ig_reusable'].sum(), f"({100*inv['ig_reusable'].mean():.1f}%)")
print('Need new IG:', (~inv['ig_reusable']).sum())
print(inv.groupby(['model','tier']).size())



✓ fn_inventory.csv: 608 FN events
Reusable: 92 (15.1%)
Need new IG: 516
model            tier    
convnextv2_tiny  Critical     34
                 Mild        141
                 Moderate     42
densenet121      Critical     31
                 Mild        126
                 Moderate     53
swinb_lora       Critical     33
                 Mild        118
                 Moderate     30
dtype: int64


In [8]:
def load_top50_mask(path_str):
    p = Path(str(path_str).replace('/content/drive/MyDrive/cxr_faithfulness', str(ROOT)))
    if not p.exists():
        p = Path(path_str)  # try raw path from manifest
    assert p.exists(), f'Missing top50: {p}'
    m = np.load(str(p))
    if m.ndim == 3:
        m = m.squeeze()
    # binary {0,1} or continuous — binarize if needed
    if set(np.unique(m)).issubset({0, 1}):
        return m.astype(np.uint8), None
    attr = normalize_ig_map(m.astype(np.float32))
    return top_mass_mask(attr), attr

def audit_row(row, attr_map=None, top50=None, ig_source='reused_nb04'):
    gt = {'xmin': row['gt_xmin'], 'ymin': row['gt_ymin'],
          'xmax': row['gt_xmax'], 'ymax': row['gt_ymax']}
    if top50 is None:
        top50, attr_map = load_top50_mask(row['nb04_top50_path'])
    if attr_map is None:
        attr_map = top50.astype(np.float32)
    peak = peak_in_box(attr_map, gt)
    mass = mass_in_box(top50, gt)
    return {
        **{k: row[k] for k in ['image_id','model','missed_class','tier',
                                'prob_miss','threshold','margin_below_thr','ig_reusable']},
        'ig_source': ig_source,
        'peak_in_box': bool(peak),
        'mass_in_box': round(mass, 4),
        'fn_bucket': fn_bucket(peak, mass),
        'ig_steps_used': np.nan,
        'ig_delta': np.nan,
    }



In [9]:
partial_path = RESULTS_PATH / 'fn_attribution_partial.csv'
done_keys = set()
audit_rows = []
n_skipped_reusable = 0

if partial_path.exists():
    _p = pd.read_csv(partial_path)
    audit_rows = _p.to_dict('records')
    done_keys = set(zip(_p['image_id'].astype(str), _p['model'].astype(str), _p['missed_class'].astype(str)))
    print(f'Resumed partial: {len(done_keys)} rows — skipping reusable pass')
else:
    reusable = inv[inv['ig_reusable']].copy()
    for _, row in reusable.iterrows():
        try:
            audit_rows.append(audit_row(row))
        except Exception as e:
            warnings.warn(f"Skip reusable {row['image_id']}/{row['model']}/{row['missed_class']}: {e}")
            n_skipped_reusable += 1
    print(f'✓ Audited reusable: {len(audit_rows)} / {len(reusable)}')
    # Checkpoint the reusable rows so we don't process them again on disconnect
    pd.DataFrame(audit_rows).to_csv(partial_path, index=False)

need_ig = inv[~inv['ig_reusable']].copy()

def fn_map_paths(image_id, model, missed_class):
    slug = class_slug(missed_class)
    base = f'{image_id}_{model}_{slug}_fn'
    return {
        'ig_path': IG_FN_PATH / f'{base}_ig.npy',
        'top50_path': IG_FN_TOP50 / f'{base}_top50.npy',
    }

def run_ig_for_missed(model, img_tensor, missed_class):
    target_idx = LABEL_COLS.index(missed_class)
    attr_norm, init_d, final_d, steps, rerun = compute_ig_with_convergence_check(
        model, img_tensor, target_idx)
    top50 = top_mass_mask(attr_norm)
    return attr_norm, top50, steps, final_d

set_seed(RANDOM_SEED)

for model_name in MODEL_NAMES:
    sub = need_ig[need_ig['model'] == model_name]
    if sub.empty:
        continue
    print(f'=== GPU: {model_name} — {len(sub)} new IG rows ===')
    model = load_model(model_name, MODELS_PATH, selected_rank, NUM_CLASSES)
    n_new = 0

    for _, row in sub.iterrows():
        key = (str(row['image_id']), str(row['model']), str(row['missed_class']))
        if key in done_keys:
            continue

        paths = fn_map_paths(*key)
        gt = {'xmin': row['gt_xmin'], 'ymin': row['gt_ymin'],
              'xmax': row['gt_xmax'], 'ymax': row['gt_ymax']}

        # resume from disk if maps exist
        if paths['top50_path'].exists() and paths['ig_path'].exists():
            attr = normalize_ig_map(np.load(paths['ig_path']))
            top50 = np.load(paths['top50_path'])
            if top50.ndim == 3: top50 = top50.squeeze()
            steps, final_d = np.nan, np.nan
        else:
            img_p = IMAGES_PATH / f"{row['image_id']}.png"
            img = cv2.imread(str(img_p))
            assert img is not None, f'Missing image {img_p}'
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_t = test_transforms(img).unsqueeze(0).cuda()

            attr, top50, steps, final_d = run_ig_for_missed(model, img_t, row['missed_class'])
            np.save(paths['ig_path'], attr)
            np.save(paths['top50_path'], top50.astype(np.uint8))
            n_new += 1

        peak = peak_in_box(attr, gt)
        mass = mass_in_box(top50, gt)
        audit_rows.append({
            'image_id': row['image_id'], 'model': row['model'],
            'missed_class': row['missed_class'], 'tier': row['tier'],
            'prob_miss': row['prob_miss'], 'threshold': row['threshold'],
            'margin_below_thr': row['margin_below_thr'],
            'ig_reusable': False,
            'ig_source': 'generated_fn',
            'peak_in_box': bool(peak),
            'mass_in_box': round(mass, 4),
            'fn_bucket': fn_bucket(peak, mass),
            'ig_steps_used': steps,
            'ig_delta': final_d,
        })
        done_keys.add(key)

        if len(audit_rows) % SAVE_EVERY == 0:
            pd.DataFrame(audit_rows).to_csv(partial_path, index=False)
            print(f'  checkpoint {len(audit_rows)} rows')

    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'  {model_name}: {n_new} new IG maps saved')

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(RESULTS_PATH / 'fn_attribution_audit.csv', index=False)
if partial_path.exists():
    partial_path.unlink()
print(f'✓ fn_attribution_audit.csv: {len(audit_df)} rows')
print(audit_df['fn_bucket'].value_counts())
print(audit_df.groupby('ig_source').size())



Resumed partial: 450 rows — skipping reusable pass
=== GPU: densenet121 — 171 new IG rows ===
✅ densenet121 loaded from densenet121_finetuned.pt
  densenet121: 0 new IG maps saved
=== GPU: convnextv2_tiny — 191 new IG rows ===
✅ convnextv2_tiny loaded from convnextv2_tiny_finetuned.pt
  convnextv2_tiny: 0 new IG maps saved
=== GPU: swinb_lora — 154 new IG rows ===


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


✅ swinb_lora loaded from swinb_lora_finetuned.pt
  checkpoint 475 rows
  checkpoint 500 rows
  checkpoint 525 rows
  checkpoint 550 rows
  checkpoint 575 rows
  checkpoint 600 rows
  swinb_lora: 154 new IG maps saved
✓ fn_attribution_audit.csv: 608 rows
fn_bucket
blind        345
attentive    263
Name: count, dtype: int64
ig_source
generated_fn    516
reused_nb04      92
dtype: int64


In [10]:
# Per model × class
summary = (
    audit_df.groupby(['model', 'missed_class', 'tier'])
    .agg(n=('fn_bucket','count'),
         n_attentive=('fn_bucket', lambda s: (s=='attentive').sum()),
         n_blind=('fn_bucket', lambda s: (s=='blind').sum()),
         mean_mass=('mass_in_box','mean'),
         mean_prob_miss=('prob_miss','mean'))
    .reset_index()
)
summary['pct_attentive'] = (summary['n_attentive'] / summary['n']).round(3)
summary.to_csv(RESULTS_PATH / 'fn_bucket_summary.csv', index=False)

# Sensitivity: MASS_TAU 0.05 / 0.10 / 0.20
sens_rows = []
for tau in [0.05, 0.10, 0.20]:
    tmp = audit_df.copy()
    tmp['fn_bucket'] = tmp.apply(
        lambda r: fn_bucket(r['peak_in_box'], r['mass_in_box'], tau=tau), axis=1)
    g = tmp.groupby('model')['fn_bucket'].value_counts(normalize=True).unstack(fill_value=0)
    g['tau'] = tau
    sens_rows.append(g.reset_index())
pd.concat(sens_rows).to_csv(RESULTS_PATH / 'fn_sensitivity_tau.csv', index=False)

print('✓ fn_bucket_summary.csv + fn_sensitivity_tau.csv')



✓ fn_bucket_summary.csv + fn_sensitivity_tau.csv


In [11]:
sev = (
    audit_df.groupby(['tier', 'model'])
    .agg(n_fn=('fn_bucket','count'),
         n_attentive=('fn_bucket', lambda s: (s=='attentive').sum()),
         n_blind=('fn_bucket', lambda s: (s=='blind').sum()))
    .reset_index()
)
sev['pct_blind'] = (sev['n_blind'] / sev['n_fn']).round(3)
sev['tier'] = pd.Categorical(sev['tier'], categories=TIER_ORDER, ordered=True)
sev = sev.sort_values(['tier','model'])
sev.to_csv(RESULTS_PATH / 'fn_by_severity.csv', index=False)

# Nodule/Mass callout
nm = audit_df[audit_df['missed_class'] == 'Nodule/Mass']
nm.to_csv(RESULTS_PATH / 'fn_nodule_mass_cases.csv', index=False)

# Figure: stacked bar per tier × model
colors = {'attentive':'#2a9d8f', 'blind':'#e76f51'}
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TIER_ORDER))
w = 0.25

for i, model in enumerate(MODEL_NAMES):
    sub = sev[sev['model']==model].set_index('tier').reindex(TIER_ORDER, fill_value=0)
    att = sub['n_attentive'].values
    bli = sub['n_blind'].values
    ax.bar(x + (i-1)*w, att, w, label=f'{model} attentive', color=colors['attentive'], alpha=0.7+i*0.1)
    ax.bar(x + (i-1)*w, bli, w, bottom=att, label=f'{model} blind' if i==0 else None,
           color=colors['blind'], alpha=0.7+i*0.1)

ax.set_xticks(x); ax.set_xticklabels(TIER_ORDER)
ax.set_ylabel('FN count'); ax.set_title('Figure FN — Attentive vs Blind by Severity Tier')
ax.legend(fontsize=8, ncol=2)
fig.savefig(FIGPATH / 'figure_fn_severity_stacked.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('✓ fn_by_severity.csv + figure_fn_severity_stacked.png')



✓ fn_by_severity.csv + figure_fn_severity_stacked.png


In [12]:
expected = [
    'fn_inventory.csv', 'fn_attribution_audit.csv', 'fn_bucket_summary.csv',
    'fn_by_severity.csv', 'fn_sensitivity_tau.csv', 'fn_nodule_mass_cases.csv',
]
ok = True
for f in expected:
    e = (RESULTS_PATH / f).exists()
    print(f"  {'✓' if e else '✗'} {f}")
    ok &= e
print(f"  {'✓' if (FIGPATH/'figure_fn_severity_stacked.png').exists() else '✗'} figure_fn_severity_stacked.png")

audit = pd.read_csv(RESULTS_PATH / 'fn_attribution_audit.csv')
inv2  = pd.read_csv(RESULTS_PATH / 'fn_inventory.csv')

# Handle dropped paths from reusable safely
n_skipped = len(inv2) - len(audit)
assert n_skipped >= 0, f"Audit unexpectedly larger than inventory"
assert len(audit) >= len(inv2) * 0.95, f'Row mismatch too large: audit={len(audit)} inv={len(inv2)} skipped={n_skipped}'

assert audit['fn_bucket'].isin(['attentive','blind']).all()
assert audit['ig_source'].isin(['reused_nb04','generated_fn']).all()
assert audit.groupby('tier').size().shape[0] >= 2, 'Need >=2 severity tiers with FNs'
print('✓ NB09 FN audit complete.' if ok else '✗ missing outputs')


  ✓ fn_inventory.csv
  ✓ fn_attribution_audit.csv
  ✓ fn_bucket_summary.csv
  ✓ fn_by_severity.csv
  ✓ fn_sensitivity_tau.csv
  ✓ fn_nodule_mass_cases.csv
  ✓ figure_fn_severity_stacked.png
✓ NB09 FN audit complete.
